In [1]:
import imperial_materials_simulation as ims
import numpy as np
import time

# Legacy Tests

This code makes sure the current physics module gives the same results as the course's original implementations.

In [2]:
n_atoms = 25
equilibrium_bond_length = 1.53
spring_constant = 15.18
sigma = 4.5
epsilon = 0.00485678
positions = np.zeros(shape=(n_atoms, 3))
positions[:, 1] = np.linspace(0, (equilibrium_bond_length+np.random.rand()/10)-1, num=n_atoms)

### Bonding Forces

In [3]:
def get_bonding_energy(pos, spring_constant, equilibrium_bondlength, calculate_force=True):
    """
    Calculate bonding potential energy and forces.
    
    The bonding potential is a harmonic spring:
        U_bond = (K/2) * sum_i (|r_{i+1} - r_i| - a)^2
    
    where K is the spring constant and a is the equilibrium bond length.
    """
    force_bond = np.zeros_like(pos)
    
    # Calculate all bond vectors at once: bond[i] = pos[i+1] - pos[i]
    bond_vectors = pos[1:] - pos[:-1]
    bond_lengths = np.linalg.norm(bond_vectors, axis=1)
    bond_lengths = np.maximum(bond_lengths, 1e-10)  # Avoid division by zero
    delta_lengths = bond_lengths - equilibrium_bondlength
    
    # Potential energy: U = (K/2) * sum(delta^2)
    potential_bond = 0.5 * spring_constant * np.sum(delta_lengths ** 2)
    
    if calculate_force:
        bond_directions = bond_vectors / bond_lengths[:, np.newaxis]
        force_magnitudes = spring_constant * delta_lengths
        force_contributions = force_magnitudes[:, np.newaxis] * bond_directions
        force_bond[:-1] += force_contributions
        force_bond[1:] -= force_contributions
    
    return potential_bond, force_bond

In [4]:
new_forces, new_potential = ims.physics.get_bonding_interactions(positions, equilibrium_bond_length, spring_constant)
old_potential, old_forces = get_bonding_energy(positions, spring_constant, equilibrium_bond_length)
assert np.allclose(new_potential, old_potential)
assert np.allclose(new_forces, old_forces)

In [6]:
start = time.perf_counter()
for i in range(1_000):
    get_bonding_energy(positions, spring_constant, equilibrium_bond_length)
old_run_time_b = time.perf_counter() - start
old_run_time_b

0.029678299993975088

In [7]:
start = time.perf_counter()
for i in range(1_000):
    ims.physics.get_bonding_interactions(positions, equilibrium_bond_length, spring_constant)
new_run_time_b = time.perf_counter() - start
new_run_time_b

0.0036694000009447336

In [8]:
old_run_time_b / new_run_time_b

8.08805253892572

### Non-Bonding Forces

In [11]:
def get_nonbonding_energy(pos, epsilon, sigma, cutoff_factor=0, calculate_force=True):
    """
    Calculate non-bonding (Lennard-Jones) potential energy and forces.
    
    The Lennard-Jones potential between atoms i and j is:
        U_LJ = epsilon * [(sigma/r_ij)^12 - 2*(sigma/r_ij)^6]
    
    This gives a minimum of -epsilon at r_ij = sigma.
    
    Non-bonding interactions are excluded for first and second neighbors
    (atoms separated by 1 or 2 bonds), as these contributions are assumed
    to be included in the bonding terms.
    
    If cutoff_factor > 0, interactions beyond cutoff_factor * sigma are ignored.
    """
    n_atoms = len(pos)
    force_nonbond = np.zeros_like(pos)
    potential_nonbond = 0.0
    
    sigma_sq = sigma * sigma
    force_prefactor = -12.0 * epsilon / sigma_sq
    
    # Cutoff setup
    use_cutoff = cutoff_factor > 0
    if use_cutoff:
        cutoff_sq = (cutoff_factor * sigma) ** 2
    
    for i in range(n_atoms - 3):
        disp = pos[i+3:] - pos[i]
        dist_sq = np.sum(disp * disp, axis=1)
        
        # Apply cutoff if enabled
        if use_cutoff:
            within_cutoff = dist_sq < cutoff_sq
            if not np.any(within_cutoff):
                continue
            disp = disp[within_cutoff]
            dist_sq = dist_sq[within_cutoff]
            j_indices = np.arange(i+3, n_atoms)[within_cutoff]
        else:
            j_indices = np.arange(i+3, n_atoms)
        
        inv_dist_sq = sigma_sq / dist_sq
        inv_dist_6 = inv_dist_sq ** 3
        inv_dist_12 = inv_dist_6 * inv_dist_6
        pair_potentials = epsilon * (inv_dist_12 - 2.0 * inv_dist_6)
        potential_nonbond += np.sum(pair_potentials)
        
        if calculate_force:
            force_factor = force_prefactor * inv_dist_sq * (inv_dist_12 - inv_dist_6)
            dforce = force_factor[:, np.newaxis] * disp
            force_nonbond[i] += np.sum(dforce, axis=0)
            force_nonbond[j_indices] -= dforce
    
    return potential_nonbond, force_nonbond

In [12]:
new_forces, new_potential = ims.physics.get_non_bonding_interactions(positions, epsilon, sigma)
old_potential, old_forces = get_nonbonding_energy(positions, epsilon, sigma)
assert np.allclose(new_potential, old_potential)
assert np.allclose(new_forces, old_forces)

In [13]:
start = time.perf_counter()
for i in range(1_000):
    get_nonbonding_energy(positions, epsilon, sigma)
old_run_time_nb = time.perf_counter() - start
old_run_time_nb

0.6932093999930657

In [14]:
start = time.perf_counter()
for i in range(1_000):
    ims.physics.get_non_bonding_interactions(positions, epsilon, sigma)
new_run_time_nb = time.perf_counter() - start
new_run_time_nb

0.0537287000042852

In [15]:
old_run_time_nb / new_run_time_nb

12.902031873798881

### Metropolis Monte Carlo Force Tracker

This just compares how much faster it is to only recalculate changed forces as compared to recalculating all forces.

In [16]:
class OldPotentialEnergyTracker():
    '''
    Utility class for efficiently tracking how total potential energy (bonding + Lennard Jones non-bonding) changes
    when a single atom is displaced.
    '''

    def __init__(self, positions: np.ndarray, epsilon: float, sigma: float, equilibrium_bond_length: float,
                spring_constant: float) -> None:
        'calculates key distances, lengths and total potential energy'
        self.positions = positions
        self.epsilon = epsilon
        self.sigma = sigma
        self.equilibrium_bond_length = equilibrium_bond_length
        self.spring_constant = spring_constant
        self.n_atoms = len(positions)

        # (n_atoms, n_atoms, 3) array where array[i,j] is the displacement between ith atom and jth atom
        self.displacements = positions.reshape((self.n_atoms, 1, 3)) - positions.reshape((1, self.n_atoms, 3))
        
        self.col_indexes, self.row_indexes = np.meshgrid(np.arange(self.n_atoms), np.arange(self.n_atoms))
        self.bonding_mask = (self.col_indexes-self.row_indexes) == 1 #all bonding interactions
        #half of non-bonding interactions (all that is needed to calculate non bonding potential energy)
        self.non_bonding_mask = (self.col_indexes-self.row_indexes) > 2 

        #should ignore distances between atom and itself (i==j). Set to 1 to prevent 0 division errors
        self.displacements[self.col_indexes==self.row_indexes] = 1
        self.lengths = np.linalg.norm(self.displacements, axis=2)

        bonding_extensions = self.lengths[self.bonding_mask] - equilibrium_bond_length
        bonding_potential =  np.sum(spring_constant/2 * bonding_extensions**2)
        six_power = (sigma/self.lengths[self.non_bonding_mask]) ** 6
        non_bonding_potential = np.sum(epsilon * (six_power**2 - 2*six_power))
        self.total_potential_energy = bonding_potential + non_bonding_potential
        
        #used in test_displacement method. allows for each pair to only be checked once. length between ij
        #calculated, but not length between ji. using the second length when only calculating potentials is redundant
        self.relevant_interactions = self.col_indexes > self.row_indexes

    def get_total_potential_energy(self):
        '''return total potential energy of molecule'''
        return self.total_potential_energy

    def test_displacement(self, atom_index: int, displacement: np.ndarray) -> float:
        '''stores temporary new positions, displacements, and lengths. Returns change in potential energy'''
        self.new_positions = self.positions.copy()
        self.new_positions[atom_index] += displacement

        self.new_displacements = self.displacements.copy()
        self.new_displacements[:, atom_index] = self.new_positions - self.new_positions[atom_index].reshape(1, 3)
        self.new_displacements[atom_index, :] = self.new_positions[atom_index].reshape(1, 3) - self.new_positions

        affected_interactions = (self.col_indexes == atom_index) | (self.row_indexes == atom_index)
        update_mask = self.relevant_interactions & affected_interactions
        self.new_lengths = self.lengths.copy()
        #axis=1 as slicing 3D array with boolean mask returns 2D array
        self.new_lengths[update_mask] = np.linalg.norm(self.new_displacements[update_mask], axis=1)

        bonding_extensions = self.new_lengths[self.bonding_mask] - self.equilibrium_bond_length
        bonding_potential =  np.sum(self.spring_constant/2 * bonding_extensions**2)
        six_power = (self.sigma/self.new_lengths[self.non_bonding_mask])**6
        non_bonding_potential = np.sum(self.epsilon * (six_power**2 - 2*six_power))
        self.new_total_potential_energy = bonding_potential + non_bonding_potential
        return self.new_total_potential_energy - self.total_potential_energy
    
    def accept_last_displacement(self) -> None:
        '''replaces internal positions, displacements, and lengths with values from last test displacement'''
        self.positions = self.new_positions
        self.displacements = self.new_displacements
        self.lengths = self.new_lengths
        self.total_potential_energy = self.new_total_potential_energy

In [17]:
F_b, PE_b = ims.physics.get_bonding_interactions(positions, equilibrium_bond_length, spring_constant)
F_nb, PE_nb = ims.physics.get_non_bonding_interactions(positions, epsilon, sigma)
true_PE = PE_b + PE_nb

energy_tracker = ims.physics.PotentialEnergyTracker(positions, epsilon, sigma, equilibrium_bond_length, spring_constant)
new_PE = energy_tracker.get_total_potential_energy()

assert np.allclose(true_PE, new_PE)

In [18]:
atom_index, displacement = 1, np.array([0.1, 1, -0.3])
new_positions = positions.copy()
new_positions[atom_index] += displacement

F_b, PE_b = ims.physics.get_bonding_interactions(new_positions, equilibrium_bond_length, spring_constant)
F_nb, PE_nb = ims.physics.get_non_bonding_interactions(new_positions, epsilon, sigma)
true_displaced_PE = PE_b + PE_nb
true_PE_change = true_displaced_PE - true_PE

new_PE_change = energy_tracker.test_displacement(atom_index, displacement)

assert np.allclose(true_PE_change, new_PE_change)

In [19]:
energy_tracker = OldPotentialEnergyTracker(positions, epsilon, sigma, equilibrium_bond_length, spring_constant)
start = time.perf_counter()
for i in range(5_000):
    energy_tracker.test_displacement(atom_index, displacement)
old_MMC_runtime = time.perf_counter() - start
old_MMC_runtime

0.2724997000186704

In [20]:
energy_tracker = ims.physics.PotentialEnergyTracker(positions, epsilon, sigma, equilibrium_bond_length, spring_constant)
start = time.perf_counter()
for i in range(5_000):
    energy_tracker.test_displacement(atom_index, displacement)
new_MMC_runtime = time.perf_counter() - start
new_MMC_runtime

0.03254809998907149

In [21]:
old_MMC_runtime / new_MMC_runtime

8.372215278623521